# Evaluate Three Extraction Methods on the Same Paper

Runs **Direct LLM**, **Static Workflow**, and **MAS** on the same paper selected by `study_id` (matching `Study#` in annotation),
all using the `wopke_100` intercropping standard with structured Pydantic output.

Includes number/table highlighting pre-processing and ROUGE-L evaluation against ground truth.

In [1]:
import os
import sys
sys.path.insert(0, '..')

import pandas as pd

from src.experimentutils import (
    get_all_paper_paths,
    read_paper_text,
    convert_pdf_to_markdown,
    convert_all_pdfs_to_markdown,
    save_extraction_results_with_timestamp,
    get_method_output_path,
    ProgressOrchestrator,
    load_ground_truth,
    build_study_paper_mapping,
    highlight_numbers_and_tables,
    build_extraction_context,
)
from src.context import create_context
from src.standards import METADATA_STANDARDS
from src.core.schema_factory import SchemaFactory
from src.direct_llm_call import extract_meta_analysis
from src.static_workflow import run_two_step_text_to_dataset

### PDF to Markdown setup (run once)

`data/wopke_100/paper_output/` is **gitignored** and must be populated locally.
Run the cell below once to convert the paper for the selected study.

In [2]:
# Build mapping from annotation Study# to paper file paths
gt_df   = load_ground_truth()
mapping = build_study_paper_mapping(gt_df)
print(f'Mapped {len(mapping)}/100 studies to paper files')

Mapped 86/100 studies to paper files


In [4]:
# Select which study to evaluate
study_id = 3  # <- Change this to test different papers

if study_id not in mapping:
    print(f'Study# {study_id} could not be mapped to a paper file!')
    print('Available study IDs:', sorted(mapping.keys()))
else:
    paper_path = mapping[study_id]
    paper_name = paper_path.split('/')[-1]
    print(f'Study# {study_id} -> {paper_name}')

    # Convert PDF to markdown if needed
    if paper_path.endswith('.pdf'):
        from src.experimentutils import convert_pdf_to_markdown
        paper_path = convert_pdf_to_markdown(paper_path)
        print(f'Converted to: {paper_path.split("/")[-1]}')

Study# 3 -> Bulson 1997 Effects of plant density on intercropped wheat and field beans in an organic farming system.md


In [5]:
# Standard and schema
standard  = METADATA_STANDARDS['wopke_100']
n_fields  = len(SchemaFactory()._parse_schema_string(standard))
n         = study_id  # alias for filename convention
print(f'Standard : wopke_100 ({n_fields} fields)')

# Ground truth for this study
gt_paper = gt_df[gt_df['Study#'] == study_id].reset_index(drop=True)
print(f'GT rows  : {len(gt_paper)}')

# Read and highlight paper text
raw_text      = read_paper_text(paper_path)
highlighted   = highlight_numbers_and_tables(raw_text)
context       = create_context(source=paper_path, name=f'study_{study_id}')

print(f'\nDocument length : {len(raw_text):,} chars')
print(f'Highlighted len : {len(highlighted):,} chars (numbers and tables marked)')

Standard : wopke_100 (42 fields)
GT rows  : 13

Document length : 61,496 chars
Highlighted len : 79,886 chars (numbers and tables marked)


In [6]:
# Shared Pydantic output schema
try:
    OutputSchema = SchemaFactory().create_from_standard(
        standard,
        record_class_name='WopkeRecord',
        output_class_name='WopkeOutput',
        records_key='yield_records',
    )
    print(f'OutputSchema : {OutputSchema.__name__}')
    record_cls = OutputSchema.model_fields['yield_records'].annotation.__args__[0]
    print(f'Record fields: {list(record_cls.model_fields.keys())}')
except Exception as e:
    OutputSchema = None
    print(f'Could not build OutputSchema ({e})')

OutputSchema : WopkeOutput
Record fields: ['Year of data', 'Duration of experiment', 'Experimental design', 'Sowing date 1', 'Sowing date 2', 'Harvest date 1', 'Harvest date 2', 'Lat', 'Lon', 'Crop species 1', 'Crop species 2', 'Crop type 1', 'Crop type 2', 'Intercropping pattern', 'Density ic 1', 'Density ic 2', 'Density sc 1', 'Density sc 2', 'N input SC1', 'N input SC2', 'N input IC1', 'N input IC2', 'N total in IC', 'N Unit', 'P input SC1', 'P input SC2', 'P input IC1', 'P input IC2', 'P total in IC', 'P Unit', 'K input SC1', 'K input SC2', 'K input IC1', 'K input IC2', 'K total in IC', 'K Unit', 'Data source', 'unified yield sc 1', 'unified yield sc 2', 'unified yield ic 1', 'unified yield ic 2', 'Yield unit']


In [7]:
# MAS objective for wopke_100 intercropping standard
mas_objective = f'''You are an expert agricultural meta-analysis specialist. Your task is to extract **intercropping experiment records** from scientific research papers. This meta-analysis compares crop yield under different intercropping settings versus sole cropping. Reason carefully and follow the multi-step process below.

**META-ANALYTIC SCHEMA (CRITICAL)**:
{standard}

**SCHEMA HANDLING RULES**
- Treat the JSON keys as the exact field names in every record.
- Schema descriptions are guidance only -- values must be concrete text from the paper.
- Do not introduce new keys or rename existing ones.
- Each record = one crop pair x one site x one year x one treatment combination.

**EXAMPLE RECORD** (from a Barley/Pea paper -- shows the expected level of detail):
```json
{{
  "Year of data": "1980",
  "Duration of experiment": "1 growing season",
  "Experimental design": "Split-plot",
  "Sowing date 1": null,
  "Sowing date 2": null,
  "Harvest date 1": "13-15 weeks after sowing",
  "Harvest date 2": "13-15 weeks after sowing",
  "Lat": "55.68",
  "Lon": "12.08",
  "Crop species 1": "Barley",
  "Crop species 2": "Pea",
  "Crop type 1": "Cereal",
  "Crop type 2": "Legume",
  "Intercropping pattern": "Mixed",
  "Density ic 1": "154",
  "Density ic 2": "30",
  "Density sc 1": "275",
  "Density sc 2": "63",
  "N input SC1": "0",
  "N input SC2": "0",
  "N input IC1": "0",
  "N input IC2": "0",
  "N total in IC": "0",
  "N Unit": "kg N/ha",
  "P input SC1": "30",
  "P input SC2": "30",
  "P input IC1": "30",
  "P input IC2": "30",
  "P total in IC": "30",
  "P Unit": "kg P/ha",
  "K input SC1": "50",
  "K input SC2": "50",
  "K input IC1": "50",
  "K input IC2": "50",
  "K total in IC": "50",
  "K Unit": "kg K/ha",
  "Data source": "Table 3,4",
  "unified yield sc 1": "4.12",
  "unified yield sc 2": "4.56",
  "unified yield ic 1": "3.4009",
  "unified yield ic 2": "1.3891",
  "Yield unit": "g/m2"
}}
```
Note: when a paper reports MULTIPLE years, MULTIPLE N-treatments, or MULTIPLE density levels, each combination is a SEPARATE record. A paper with 2 years x 2 N-levels = 4 records for the same crop pair.

**EXTRACTION PROCESS**

**Step 1: Label the Original Document**
Read the complete paper. Add XML tags around values matching schema fields:
  <Crop_species_1>maize</Crop_species_1>, <unified_yield_ic_1>6.1</unified_yield_ic_1>
  <Density_ic_1>5</Density_ic_1>, <N_input_SC1>120</N_input_SC1>
Text marked with <<NUM: ... >> already highlights numerical values -- pay close attention to these.
Text between <<TABLE_START>> and <<TABLE_END>> markers likely contains yield and treatment data in tabular form.
Preserve all original text -- only insert tags inline.

**Step 2: Extract records from tables and results**
Focus especially on sections marked <<TABLE_START>> ... <<TABLE_END>> and on Results / Discussion.
Each row in a yield table or each treatment combination that reports both intercrop AND sole-crop yields defines one record.
For every yield anchor, extract ALL associated schema fields:
- `Year of data`: year(s) data collected -- check Title, Abstract, Site description.
- `Lat` / `Lon`: site coordinates -- check Site description; infer from city/country if not stated.
- `Crop species 1` / `Crop species 2`: common or scientific name.
- `Crop type 1` / `Crop type 2`: cereal, legume, oilseed, root crop -- infer from species.
- `Intercropping pattern`: Row, Strip, or Mixed -- check Methods.
- `Density ic/sc 1/2`: planting densities -- check Planting/Management/Table.
- `N/P/K input SC1/SC2/IC1/IC2`, totals, units: fertilizer data -- check Fertilization/Table.
- `unified yield sc/ic 1/2` and `Yield unit`: the core yield data.
- `Data source`: which table or figure the data comes from.

**Step 3: Complete and validate**
- Site constants (Year, Lat, Lon, Experimental design, Intercropping pattern) uniform across treatments MUST be in every record.
- Nutrient inputs and densities that are uniform across treatments MUST be filled into every record.
- If a value genuinely cannot be found, set to null. **Do NOT fabricate values.**
- **Do NOT create extra records** beyond what the data reports. But ensure EVERY distinct treatment x year x crop-pair combination gets its own record.

**Step 4: Finalize**
One JSON entry per record using exact schema keys. Keep original units. De-duplicate exact duplicates.

**OUTPUT FORMAT**: JSON under key `yield_records`. Each record must use exact schema field names. If no data found, return empty array.'''

---
## Method 1 -- Direct LLM

Uses the highlighted text for better number visibility.

In [8]:
result_direct = extract_meta_analysis(paper_path, schema=standard)
print(f'Extracted {len(result_direct.yield_records)} record(s) (GT has {len(gt_paper)})')

Extracted 1 record(s) (GT has 13)


In [10]:
df_direct = pd.DataFrame([r.model_dump() for r in result_direct.yield_records])
df_direct

,Year of data,Duration of experiment,Experimental design,Sowing date 1,Sowing date 2,Harvest date 1,Harvest date 2,Lat,Lon,Crop species 1,...,K input IC1,K input IC2,K total in IC,K Unit,Data source,unified yield sc 1,unified yield sc 2,unified yield ic 1,unified yield ic 2,Yield unit
0,1987–1988,1 growing season,Randomized Complete Block Design (RCBD),5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,Triticum aestivum,...,0,0,0,kg K2O ha−1,Table 1,3.7,3.7,None,None,t ha−1


In [11]:
path_direct = save_extraction_results_with_timestamp(
    results=result_direct,
    base_name=f'{n}_direct_llm_{n_fields}fields',
    records_key='yield_records',
    include_time=False,
)
print(f'Saved: {path_direct}')

Saved: /home/com3dian/Github/meta_analysis_agents/outputs/2026-03-11/3_direct_llm_42fields_2026-03-11.csv


---
## Method 2 -- Static Workflow (Facts -> Schema Dataset)

Also uses highlighted text.

In [12]:
out_workflow = run_two_step_text_to_dataset(
    text=highlighted,
    max_facts=40,
    use_llm_dataset_builder=True,
    dataset_standard=standard,
    dataset_records_key='yield_records',
)

print(f"Facts extracted  : {len(out_workflow['facts_result'].facts)}")
print(f"Records built    : {len(out_workflow['dataset'])} (GT has {len(gt_paper)})")
print(f"Validation OK    : {out_workflow['validation_ok']}")
if out_workflow['validation_issues']:
    print(f"Validation issues: {out_workflow['validation_issues']}")

Facts extracted  : 20
Records built    : 1 (GT has 13)
Validation OK    : False
Validation issues: ["Missing required columns: ['subject', 'predicate', 'object']"]


In [13]:
df_workflow = out_workflow['dataset']
df_workflow

,Year of data,Duration of experiment,Experimental design,Sowing date 1,Sowing date 2,Harvest date 1,Harvest date 2,Lat,Lon,Crop species 1,...,K input IC1,K input IC2,K total in IC,K Unit,Data source,unified yield sc 1,unified yield sc 2,unified yield ic 1,unified yield ic 2,Yield unit
0,1987–1988,None,factorial experiment,None,None,None,None,None,None,Wheat,...,None,None,None,None,"Table 1,2; Figure 3; Supplementary Table S1",None,None,None,None,None


In [14]:
if 'schema_output' in out_workflow:
    path_workflow = save_extraction_results_with_timestamp(
        results=out_workflow['schema_output'],
        base_name=f'{n}_static_workflow_{n_fields}fields',
        records_key='yield_records',
        include_time=False,
    )
    print(f'Saved: {path_workflow}')
else:
    path_workflow = None
    print('No schema_output found -- run with use_llm_dataset_builder=True.')

Saved: /home/com3dian/Github/meta_analysis_agents/outputs/2026-03-11/3_static_workflow_42fields_2026-03-11.csv


---
## Method 3 -- MAS (Multi-Agent System)

Uses highlighted text in context and the detailed extraction objective.

In [15]:
orchestrator = ProgressOrchestrator(topology_name='default')

result_mas = orchestrator.run(
    source=context,
    objective=mas_objective,
    output_schema=OutputSchema,
)
print(f'Success: {result_mas.success}')
print(f'Steps completed: {result_mas.steps_completed}/{result_mas.plan_steps_count}')

Generating plan:   0%|          | 0/1 [00:00<?, ?plan/s]

Executing plan:   0%|          | 0/5 [00:00<?, ?step/s]

  initializing:   0%|          | 0/6 [00:00<?, ?phase/s]

Success: True
Steps completed: 5/5


In [16]:
workspace = result_mas.final_workspace
raw_mas = workspace.get('final_meta_analysis_records', {})

if isinstance(raw_mas, dict):
    records_mas = raw_mas.get('yield_records', raw_mas.get('records', []))
elif isinstance(raw_mas, list):
    records_mas = raw_mas
else:
    records_mas = []

print(f'Extracted {len(records_mas)} record(s) (GT has {len(gt_paper)})')
df_mas = pd.DataFrame(records_mas)
df_mas

Extracted 16 record(s) (GT has 13)


,Year of data,Duration of experiment,Experimental design,Sowing date 1,Sowing date 2,Harvest date 1,Harvest date 2,Lat,Lon,Crop species 1,...,K input IC1,K input IC2,K total in IC,K Unit,Data source,unified yield sc 1,unified yield sc 2,unified yield ic 1,unified yield ic 2,Yield unit
0,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None
1,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None
2,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None
3,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None
4,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None
5,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None
6,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None
7,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None
8,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None
9,1987-1988,None,randomized block design,5 November 1987,4 November 1987,3 October 1988,3 October 1988,None,None,wheat (Triticum aestivum),...,None,None,None,None,Bulson 1997,None,None,None,None,None


In [ ]:
path_mas = save_extraction_results_with_timestamp(
    results=raw_mas,
    base_name=f'{n}_mas_{n_fields}fields',
    records_key='yield_records',
    include_time=False,
)
print(f'Saved: {path_mas}')

---
## Summary

In [ ]:
summary = pd.DataFrame([
    {'method': 'direct_llm',      'records': len(result_direct.yield_records), 'output': path_direct},
    {'method': 'static_workflow', 'records': len(df_workflow),                  'output': path_workflow or 'not saved'},
    {'method': 'mas',             'records': len(records_mas),                  'output': path_mas},
])
summary

---
## Evaluation Against Ground Truth (ROUGE-L)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rouge_l_score(reference, hypothesis):
    ref = str(reference).lower().split()
    hyp = str(hypothesis).lower().split()
    if not ref or not hyp:
        return 0.0
    m, nl = len(ref), len(hyp)
    dp = [[0]*(nl+1) for _ in range(m+1)]
    for i in range(1,m+1):
        for j in range(1,nl+1):
            dp[i][j] = dp[i-1][j-1]+1 if ref[i-1]==hyp[j-1] else max(dp[i-1][j],dp[i][j-1])
    lcs = dp[m][nl]
    p, r = lcs/nl, lcs/m
    return 0.0 if p+r==0 else 2*p*r/(p+r)

def best_match_rouge_l(ext_df, gt_df, shared_cols):
    results = []
    for _, ext_row in ext_df.iterrows():
        best_mean, best_scores = -1.0, {}
        for _, gt_row in gt_df.iterrows():
            cs = {c: rouge_l_score(str(ext_row.get(c,'')), str(gt_row.get(c,''))) for c in shared_cols}
            m = float(np.mean(list(cs.values())))
            if m > best_mean:
                best_mean, best_scores = m, cs
        results.append({f'rougeL_{c}':v for c,v in best_scores.items()})
    return pd.DataFrame(results)

print('ROUGE-L utilities ready.')

In [ ]:
wopke_field_names = list(SchemaFactory()._parse_schema_string(standard).keys())
gt_cols_set   = set(gt_paper.columns)
shared_fields = [f for f in wopke_field_names if f in gt_cols_set]
print(f'Shared evaluation fields: {len(shared_fields)} / {len(wopke_field_names)}')

In [ ]:
method_paths = {
    'direct_llm':      path_direct,
    'static_workflow': path_workflow,
    'mas':             path_mas,
}
method_scores = {}

for method, csv_path in method_paths.items():
    if csv_path is None or not os.path.exists(str(csv_path)):
        print(f'[{method}] no output CSV -- skipping.')
        continue
    ext_df = pd.read_csv(csv_path)
    if ext_df.empty:
        print(f'[{method}] empty CSV -- skipping.')
        continue
    eval_cols = [c for c in shared_fields if c in ext_df.columns]
    if not eval_cols:
        print(f'[{method}] no shared columns -- skipping.')
        continue
    scores_df = best_match_rouge_l(ext_df, gt_paper, eval_cols)
    method_scores[method] = scores_df
    print(f'[{method}] {len(ext_df)} rows, {len(eval_cols)} cols -> mean ROUGE-L = {scores_df.mean().mean():.3f}')

print('\nEvaluation complete.')

In [ ]:
if not method_scores:
    print('No scores to plot.')
else:
    COLORS = {'direct_llm':'#4C78A8','static_workflow':'#F58518','mas':'#54A24B'}
    all_fields = sorted({c.replace('rougeL_','') for s in method_scores.values() for c in s.columns})
    nf, nm = len(all_fields), len(method_scores)
    bw, x = 0.8/max(nm,1), np.arange(nf)
    fig,(ax1,ax2) = plt.subplots(2,1,figsize=(max(14,nf*0.55),10))
    for i,(meth,sdf) in enumerate(method_scores.items()):
        means = [sdf[f'rougeL_{f}'].mean() if f'rougeL_{f}' in sdf.columns else float('nan') for f in all_fields]
        ax1.bar(x+i*bw, means, width=bw, label=meth, color=COLORS.get(meth,'steelblue'), alpha=0.85)
    ax1.set_xticks(x+bw*(nm-1)/2)
    ax1.set_xticklabels(all_fields, rotation=45, ha='right', fontsize=8)
    ax1.set_ylim(0,1.05); ax1.set_ylabel('Mean ROUGE-L'); ax1.legend(loc='upper right')
    ax1.set_title(f'Per-field ROUGE-L -- Study# {study_id}: {paper_name[:60]}')
    ax1.axhline(0.5, color='grey', ls='--', lw=0.8, alpha=0.6)
    mnames = list(method_scores.keys())
    omeans = [method_scores[m].mean().mean() for m in mnames]
    bars = ax2.bar(mnames, omeans, color=[COLORS.get(m,'steelblue') for m in mnames], alpha=0.85, width=0.4)
    for bar,val in zip(bars,omeans):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02, f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
    ax2.set_ylim(0,1.05); ax2.set_ylabel('Mean ROUGE-L'); ax2.set_title('Overall mean ROUGE-L by method')
    ax2.axhline(0.5, color='grey', ls='--', lw=0.8, alpha=0.6)
    plt.tight_layout()
    plot_path = get_method_output_path(n,'evaluation_plot',n_fields).replace('.csv','.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Plot saved: {plot_path}')